In [1]:
import numpy as np

In [2]:
import torch

In [3]:
MODEL_PATH = "../data/mace-omat-0-medium.model"

In [11]:
import torch

def load_mace_model(path):
    # matches MACE behavior (torch.load), but adds a safe fallback
    try:
        obj = torch.load(path, map_location="cpu", weights_only=False)
        loader = "torch.load"
    except Exception as e:
        obj = torch.jit.load(path, map_location="cpu")
        loader = f"torch.jit.load (torch.load failed: {type(e).__name__})"

    # common save formats: either nn.Module directly, or dict with key "model"
    if isinstance(obj, dict) and "model" in obj and hasattr(obj["model"], "parameters"):
        model = obj["model"]
        container = "dict['model']"
    else:
        model = obj
        container = type(obj).__name__

    return model, loader, container

model, loader, container = load_mace_model(MODEL_PATH)
print("Loaded via:", loader)
print("Container:", container)
print("Model type:", type(model))
print("torch default dtype:", torch.get_default_dtype())


Loaded via: torch.load
Container: ScaleShiftMACE
Model type: <class 'mace.modules.models.ScaleShiftMACE'>
torch default dtype: torch.float32


In [12]:
from collections import Counter

param_dtypes = Counter(p.dtype for p in model.parameters())
buf_dtypes   = Counter(b.dtype for b in model.buffers())

print("Parameter dtypes:", dict(param_dtypes))
print("Buffer dtypes:   ", dict(buf_dtypes))

# Quick sanity: show first few params with dtype/shape
for i, (n, p) in enumerate(model.named_parameters()):
    if i >= 15: break
    print(f"{i:02d} {n:60s} {str(p.dtype):12s} {tuple(p.shape)}")


Parameter dtypes: {torch.float64: 31}
Buffer dtypes:    {torch.int64: 2, torch.float64: 60}
00 node_embedding.linear.weight                                 torch.float64 (11392,)
01 interactions.0.linear_up.weight                              torch.float64 (16384,)
02 interactions.0.conv_tp_weights.layer0.weight                 torch.float64 (10, 64)
03 interactions.0.conv_tp_weights.layer1.weight                 torch.float64 (64, 64)
04 interactions.0.conv_tp_weights.layer2.weight                 torch.float64 (64, 64)
05 interactions.0.conv_tp_weights.layer3.weight                 torch.float64 (64, 512)
06 interactions.0.linear.weight                                 torch.float64 (65536,)
07 interactions.0.skip_tp.weight                                torch.float64 (5832704,)
08 interactions.0.density_fn.layer0.weight                      torch.float64 (10, 1)
09 interactions.1.linear_up.weight                              torch.float64 (32768,)
10 interactions.1.conv_tp_weights.la

In [9]:
mixed_params = len(param_dtypes) > 1
mixed_bufs   = len(buf_dtypes) > 1

print("Mixed parameter dtype?", mixed_params)
print("Mixed buffer dtype?   ", mixed_bufs)

if mixed_params or mixed_bufs:
    print("\n⚠️ Mixed dtypes in the model can trigger errors like "
          "'both inputs should have same dtype' when combined with data tensors "
          "of a different dtype.")


Mixed parameter dtype? False
Mixed buffer dtype?    True

⚠️ Mixed dtypes in the model can trigger errors like 'both inputs should have same dtype' when combined with data tensors of a different dtype.


In [4]:
# Check finetuned model

In [29]:
import os, torch
from pathlib import Path
from ase import build
from ase.io import read, write
from mace.calculators import MACECalculator

finetuned_model_path = "../data/mp-0_ft_x050_060_ef_10_1/omat-0_ft_x050_060_ef_10_1_compiled.model"
original_model_path = "../data/mace-omat-0-medium.model"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

# atoms = build.molecule('H2O')

In [43]:
atoms = build.molecule('H2O')

In [18]:
atoms.calc = MACECalculator(model_paths=[original_model_path], device="cpu")

Using head default out of ['default']
No dtype selected, switching to float64 to match model dtype.


/opt/homebrew/Caskroom/miniconda/base/envs/research/lib/python3.10/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


In [44]:
atoms.calc = MACECalculator(model_paths=[finetuned_model_path], device="cpu")

/opt/homebrew/Caskroom/miniconda/base/envs/research/lib/python3.10/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/opt/homebrew/Caskroom/miniconda/base/envs/research/lib/python3.10/site-packages/torch/serialization.py:1493: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  warnings.warn(


Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.


In [41]:
dft_path  = Path("../data/pbe_traj.extxyz")
mace_path = Path("../data/pbe_traj_w_mace.extxyz")   # (even if extension is .xyz, we read as extxyz-compatible)

dft_frames  = read(str(dft_path), ":", format="extxyz")
mace_frames = read(str(mace_path), ":", format="extxyz")

In [32]:
dft_frames[0].calc = MACECalculator(model_paths=[original_model_path], device="cpu")

Using head default out of ['default']
No dtype selected, switching to float64 to match model dtype.


/opt/homebrew/Caskroom/miniconda/base/envs/research/lib/python3.10/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


In [33]:
dft_frames[0].get_total_energy()

-17.245926745680727

In [34]:
dft_frames[0].calc = MACECalculator(model_paths=[finetuned_model_path], device="cpu")

/opt/homebrew/Caskroom/miniconda/base/envs/research/lib/python3.10/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/opt/homebrew/Caskroom/miniconda/base/envs/research/lib/python3.10/site-packages/torch/serialization.py:1493: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  warnings.warn(


Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.


In [35]:
dft_frames[0].get_total_energy()

-17.600400924682617

In [42]:
dft_frames[0].get_potential_energy()

np.float64(-17.58894461)

In [47]:
atoms.pbc = [True, True, True]

In [49]:
calc = MACECalculator(model_paths=[finetuned_model_path], device="cpu")

/opt/homebrew/Caskroom/miniconda/base/envs/research/lib/python3.10/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/opt/homebrew/Caskroom/miniconda/base/envs/research/lib/python3.10/site-packages/torch/serialization.py:1493: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  warnings.warn(


Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.


In [50]:
print("Supported atomic numbers (Z):", calc.z_table.zs)
print("Contains H (Z=1)?", 1 in calc.z_table.zs)

Supported atomic numbers (Z): [8, 11, 22, 23, 24, 25, 26, 27, 28]
Contains H (Z=1)? False


In [11]:
from pymatgen.core.structure import Structure
from glob import glob

In [12]:
exp_cif_path = '../data/experiments/orderings/'
exp_cifs = glob(exp_cif_path + "*.cif")

In [13]:
exp_cifs

['../data/experiments/orderings/4_Na2-5.cif',
 '../data/experiments/orderings/2_Na2-3.cif',
 '../data/experiments/orderings/3_Na1-2.cif',
 "../data/experiments/orderings/O'3-Na0.5MnO2.cif",
 '../data/experiments/orderings/5_Na-1-3.cif',
 '../data/experiments/orderings/Na0.5VO2.cif',
 "../data/experiments/orderings/O'3-Na0.33MnO2.cif",
 '../data/experiments/orderings/1_Na1_EntryWithCollCode85317.cif']

In [17]:
for cif in exp_cifs:
    struct = Structure.from_file(cif)
    struct.make_supercell([1, 1, 3])
    if 'V' in cif:
        name = 'cd_a.vasp'
    elif 'Mn' in cif and 'Na0.33' in cif:
        name = 'xc_a.vasp'
    elif 'Mn' in cif and 'Na0.5' in cif:
        name = 'xc_b.vasp'
    elif '5_Na' in cif:
        name = 'JMAS_a.vasp'
    elif '4_Na' in cif:
        name = 'JMAS_b.vasp'
    elif '3_Na' in cif:
        name = 'JMAS_c.vasp'
    elif '2_Na' in cif:
        name = 'JMAS_d.vasp'
    elif '1_Na' in cif:
        name = 'JMAS_e.vasp'
    struct.to(fmt="poscar", filename="../data/experiments/orderings/" + name)    

/opt/homebrew/Caskroom/miniconda/base/envs/research/lib/python3.10/site-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: Skipping relative stoichiometry check because CIF does not contain formula keys.
  struct = parser.parse_structures(primitive=primitive)[0]


In [8]:
test.to(fmt="poscar", filename="../data/experiments/orderings/JMAS_a.vasp")

'Na6 Ni6 O12\n1.0\n   4.9840028672723848    0.0000000000000000   -1.8583874889324492\n  -0.0000000000000002    2.8451000000000000    0.0000000000000002\n   0.0000000000000000    0.0000000000000000   16.7478000000000016\nNa Ni O\n6 6 12\ndirect\n   0.0000000000000000    0.4999999999999999    0.1666666666666667 Na+\n   0.0000000000000000    0.4999999999999999    0.5000000000000000 Na+\n   0.0000000000000000    0.4999999999999999    0.8333333333333333 Na+\n   0.5000000000000000    0.0000000000000000    0.1666666666666667 Na+\n   0.5000000000000000    0.0000000000000000    0.4999999999999999 Na+\n   0.5000000000000000    0.0000000000000000    0.8333333333333333 Na+\n   0.0000000000000000    0.0000000000000000    0.0000000000000000 Ni3+\n   0.0000000000000000    0.0000000000000000    0.3333333333333333 Ni3+\n   0.0000000000000000    0.0000000000000000    0.6666666666666666 Ni3+\n   0.5000000000000000    0.4999999999999999    0.0000000000000000 Ni3+\n   0.5000000000000000    0.49999999999999